In [ ]:
import optuna
from optuna.storages.journal import JournalStorage, JournalFileBackend
import pandas as pd
import os

# === CONFIGURATION ===
# Change this for each of your 4 notebooks
MODEL_TYPE = "MetaCNNLSTM"  # Options: "MetaCNNLSTM", "DeepCNNLSTM", "TST", "ContrastiveNet"

DB_DIR = "C:\\Users\\kdmen\\Repos\\pers-gest-cls\\dataset\\optuna_dbs"
FILE_NAME = f"1s3w_mamlpp100_{MODEL_TYPE}_2fcv_hpo"
JOURNAL_PATH = os.path.join(DB_DIR, f"{FILE_NAME}.log")

print(f"Loading study: {FILE_NAME}")
print(f"From path: {JOURNAL_PATH}")

Loading study: 1s3w_mamlpp100_MetaCNNLSTM_2fcv_hpo
From path: C:\Users\kdmen\Repos\pers-gest-cls\optuna_journals\1s3w_mamlpp100_MetaCNNLSTM_2fcv_hpo.log


In [ ]:
# 2. Initialize storage
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# 3. Use the TOP-LEVEL optuna function to get summaries
summaries = optuna.get_all_study_summaries(storage)

if not summaries:
    print("No studies found in this file.")
else:
    for s in summaries:
        print(f"Internal Study Name: '{s.study_name}'")

    STUDY_NAME = s.study_name

Internal Study Name: 'mamlpp_pretr_MetaCNNLSTM_2fcv_hpo'


In [3]:
# Initialize the storage backend
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# Load the study
study = optuna.load_study(
    study_name=STUDY_NAME,
    storage=storage
)

print(f"Study loaded with {len(study.trials)} trials.")
print(f"Best value (Accuracy): {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Study loaded with 103 trials.
Best value (Accuracy): 0.7979
Best Hyperparameters:
  meta_batchsize: 8
  maml_inner_steps: 5
  maml_inner_steps_eval: 15
  maml_alpha_init: 0.0006014636384157608
  maml_alpha_init_eval: 0.0035407311968777147
  outer_lr: 0.00024725128234412636
  wd: 1.696625960836635e-06
  groupnorm_num_groups: 4
  use_GlobalAvgPooling: False
  finetuning_approach: full
  best_or_last_pretr: best
  context_emb_dim: 32
  context_pool_type: mean
  episodes_per_epoch_train: 500
  optimizer: adam
  use_maml_msl: False


In [4]:
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate


In [5]:
# 1. Plot optimization history
fig_hist = plot_optimization_history(study)
fig_hist.show()


In [6]:
# 2. Plot Hyperparameter Importance
try:
    fig_imp = plot_param_importances(study)
    fig_imp.show()
except Exception as e:
    print(f"Could not plot importance (might need more trials): {e}")

In [7]:
# 3. Parallel Coordinate Plot (Visualizes high-dimensional relationships)
fig_parallel = plot_parallel_coordinate(study)
fig_parallel.show()

In [8]:
# FULL
params_to_plot = ["best_or_last_pretr", "context_emb_dim", "context_pool_type", "episodes_per_epoch_train", "finetuning_approach", "groupnorm_num_groups", 
                  "maml_inner_steps", "maml_inner_steps_eval", "maml_msl_num_epochs", "meta_batchsize", "optimizer", "use_GlobalAvgPooling", "use_maml_msl", "wd"]

# PRETRAINING STUFF / MISC
# Apparently finetuning_approach and use_maml_msl only had 1 value? Yes for finetuning currently, but use_maml_msl should have had true and false as well no? ...
params_to_plot_MISC = ["best_or_last_pretr", "finetuning_approach", "use_maml_msl"]

# ARCHITECTURE? --> What does this even mean? How can this vary if we are using the pretrained model....
params_to_plot_ARCH = ["context_emb_dim", "context_pool_type", "groupnorm_num_groups", "use_GlobalAvgPooling"]

# MAML++
params_to_plot_MAMLPP = ["maml_inner_steps", "maml_inner_steps_eval", "maml_msl_num_epochs", "use_maml_msl"]

# LRs MAML++
params_to_plot_LRS = ["maml_alpha_init", "maml_alpha_init_eval", "outer_lr"]

# Learning HPs
params_to_plot_HPS = ["episodes_per_epoch_train", "meta_batchsize", "optimizer", "wd"]

In [9]:
from optuna.visualization import plot_slice


In [10]:
fig_slice = plot_slice(study, params=params_to_plot_MISC)
fig_slice.show()

In [11]:
fig_slice = plot_slice(study, params=params_to_plot_ARCH)
fig_slice.show()

In [12]:
fig_slice = plot_slice(study, params=params_to_plot_MAMLPP)
fig_slice.show()

In [13]:
fig_slice = plot_slice(study, params=params_to_plot_LRS)
fig_slice.show()

In [14]:
fig_slice = plot_slice(study, params=params_to_plot_HPS)
fig_slice.show()

In [15]:
trials_df = study.trials_dataframe()

# Extract the custom user attributes into the dataframe
trials_df['mean_pretrain_val_acc'] = [t.user_attrs.get('mean_pretrain_val_acc') for t in study.trials]
trials_df['fold_mean_accs'] = [t.user_attrs.get('fold_mean_accs') for t in study.trials]

# Filter for relevant columns and sort by best performance
results_summary = trials_df[['number', 'value', 'mean_pretrain_val_acc', 'datetime_start', 'duration']].sort_values(by='value', ascending=False)
results_summary.head(10)

,number,value,mean_pretrain_val_acc,datetime_start,duration
101,101,0.797917,0.805208,2026-03-26 03:52:31.977438,0 days 00:22:02.850370
90,90,0.794792,0.793750,2026-03-26 02:43:03.241695,0 days 00:20:40.736692
100,100,0.794271,0.786458,2026-03-26 03:50:00.258191,0 days 00:16:02.925726
102,102,0.788542,0.784896,2026-03-26 04:04:43.755189,0 days 00:31:15.548370
87,87,0.783333,0.782292,2026-03-26 02:22:13.731006,0 days 00:20:07.122541
91,91,0.779167,0.784896,2026-03-26 02:47:37.330687,0 days 00:13:11.695092
65,65,0.779167,0.776042,2026-03-25 23:54:46.841415,0 days 00:19:31.354812
63,63,0.775521,0.776563,2026-03-25 23:44:07.661836,0 days 00:18:14.545968
86,86,0.774479,0.784896,2026-03-26 02:14:04.313026,0 days 00:19:34.446869
99,99,0.770833,0.770833,2026-03-26 03:40:22.714688,0 days 00:23:42.375277
